%pip install --user --force-reinstall "numpy<2"

%pip install --user --upgrade numexpr

!pip3 install --user xarray

!pip3 install --user netcdf4

!pip3 install --user matplotlib

In [7]:
import xarray as xr

ds = xr.open_dataset("bc_aeropt_cmip6_volc_lw_b16_sw_b14_1970.nc")
ds

<xarray.Dataset> Size: 11MB
Dimensions:      (altitude: 70, latitude: 36, month: 12, solar_bands: 14,
                  terrestrial_bands: 16)
Coordinates:
  * altitude     (altitude) float32 280B 5.0 5.5 6.0 6.5 ... 38.0 38.5 39.0 39.5
  * latitude     (latitude) float32 144B -87.5 -82.5 -77.5 ... 77.5 82.5 87.5
  * month        (month) float32 48B 1.441e+03 1.442e+03 ... 1.451e+03 1.452e+03
Dimensions without coordinates: solar_bands, terrestrial_bands
Data variables:
    wl1_sun      (solar_bands) float32 56B ...
    wl2_sun      (solar_bands) float32 56B ...
    wl1_earth    (terrestrial_bands) float32 64B ...
    wl2_earth    (terrestrial_bands) float32 64B ...
    ext_sun      (solar_bands, latitude, altitude, month) float32 2MB ...
    omega_sun    (solar_bands, latitude, altitude, month) float32 2MB ...
    g_sun        (solar_bands, latitude, altitude, month) float32 2MB ...
    ext_earth    (terrestrial_bands, latitude, altitude, month) float32 2MB ...
    omega_earth  (terrestrial_bands, latitude, altitude, month) float32 2MB ...
    g_earth      (terrestrial_bands, latitude, altitude, month) float32 2MB ...
Attributes:
    creation_date:  Thu May 24 15:26:17 CEST 2018
    frequency:      month
    Conventions:    None
    source_file:    CMIP_ECHAM6_radiation.nc
    title:          Stratospheric Aerosol forcing with tropospheric values ex...
    history:        Thu May 24 17:24:42 2018: ncks -a -d month,1440,1451 -O ....
    NCO:            "4.5.2"

In [ ]:
from pathlib import Path
import xarray as xr
import numpy as np

# 
# Settings up top so they are easy to change later if necessary
# 

folder = Path(".")

factor = 10.0
variables = ["ext_sun", "ext_earth"]

source_start_year = 1991
target_start_year = 2005

n_years = 10  # 1991-2000 -> 2005-2014 meaning we have 10 years 

# background year is year 2000
bg_year = 2000

# Over how many months the rampup should happen until full scale is reached.
ramp_up_months = 12  

# start of ramp down, and by which year the rampdown should be finished (and we're back at background levels)
ramp_down_start_year = 2008
ramp_down_end_year = 2014

tau = 2.0  # e-folding time


#
# some functions to help 
#


######## We're using the classic smoothstep function instead of a linear ramp-up, because it has the benefit of starting slow, then going faster, and ending slow.
######## otherwise it's performing the same of going from 0 to 1, as a simpler linear ramp would
######## This is just a maybe-unnecessary bonus, as a linear ramp would have done a similar job, but it's a neat addition
def smoothstep(x):                
    x = np.clip(x, 0.0, 1.0)                          #clip everything away that is outside of the bounds of 0 and 1, we don't need that
    return 3.0 * x**2 - 2.0 * x**3                    #how the classic smoothstep function is defined.


######## We need weights for the slow ramp up. It's the function implementing smoothstep over the desired timeframe
######## This function basically takes a desired amount of months and scales them to 0-1, then gives the resulting numbers to smoothstep
######## goal is weight 0 at start and weight 1 at end.
def ramp_up_weight(t_months, ramp_up_months):        #t_months is how many months have passed since the start of the rampup = since eruption, 
                                                    #ramp_up_months: how long should the rampup be?                                                                                                       
    if ramp_up_months <= 1:                         #t_months = 0 means june 2005.   
        return 1.0
                                                    #this is just so we avoid division by 0 in the next line
    x = t_months / (ramp_up_months - 1)             #-1 because t_months is 0 in month 1, 1 in month 2 .... and 11 in month 12
    return smoothstep(x)                            


######## function for the ramp_down weights so that decay to 2000 background is slow and smooth. weight 1 for start year, 0 for end year
def ramp_down_weight(tgt_year, month, start_year, end_year, tau):   #

    # january = +0/12, february = +1/12, ..., december = +11/12
    t = tgt_year + (month - 1) / 12.0    #a bit convoluted, but basically we put year and month into a single number so that we start at January 2008 and 
                                         #end by december 2014.      
    t_start = start_year
    t_end = end_year + 11.0 / 12.0  # december of end year

    if t < t_start:              #full effect before start of ramp-down
        return 1.0

    if t >= t_end:               #reach background by end
        return 0.0

    # exponential decay, normalized so that w(t_start) = 1 and w(t_end) = 0
    raw = np.exp(-(t - t_start) / tau)                #exponential decay
    raw_end = np.exp(-(t_end - t_start) / tau)

    w = (raw - raw_end) / (1.0 - raw_end)         #normalizing so we get exactly to 0 by the end, as a normal exponential function would never reach 0

    return w


#
# Main program
#
 #pathing
bg_file = folder / f"bc_aeropt_cmip6_volc_lw_b16_sw_b14_{bg_year}.nc"

with xr.open_dataset(bg_file) as bg:

    for year_offset in range(n_years):                               #loop through all years we want to copy (in a modified way) over

        src_year = source_start_year + year_offset                   #define source year and target year
        tgt_year = target_start_year + year_offset

        src_file = folder / f"bc_aeropt_cmip6_volc_lw_b16_sw_b14_{src_year}.nc"          #define source file and target file
        tgt_file = folder / f"bc_aeropt_cmip6_volc_lw_b16_sw_b14_{tgt_year}.nc"

        print(f"Bearbeite {src_year} -> {tgt_year}")

        with xr.open_dataset(src_file) as src, xr.open_dataset(tgt_file) as tgt:

            # take target file as base
            # keep everything that isn't modified as it is
            out = tgt.copy(deep=True)

            # pinatubo starts 1991 in july
            # Jan-may 1991 stay unchanged
            if src_year == 1991:
                months = np.arange(6, 13)   # june to december
            else:
                months = np.arange(1, 13)   # january to february

            for m in months:                                              ########outer loop, through months

                idx = m - 1  # january = 0, ... december = 11

                # time since start of eruption.
                # june 2005 t_months = 0.
                t_months = (tgt_year - target_start_year) * 12 + (m - 6)

                # use ramp up weight function from above
                w_up = ramp_up_weight(t_months, ramp_up_months)

                # use ramp down weight function from above
                w_down = ramp_down_weight(
                    tgt_year=tgt_year,
                    month=m,
                    start_year=ramp_down_start_year,
                    end_year=ramp_down_end_year,
                    tau=tau
                )

                # total weight
                # it goes from 0 to 1 in beginning
                # and from 1 to 0 in the end
                w_total = w_up * w_down

                effective_factor = factor * w_total          #so that we have a ramp up and ramp down from our chosen factor

                print(                                       #just printing a bit so we know the code works and what it's actually doing
                    f"  {tgt_year}-{m:02d}: "
                    f"w_up={w_up:.3f}, "
                    f"w_down={w_down:.3f}, "
                    f"effective_factor={effective_factor:.3f}"
                )

                for var in variables:                           #############inner loop, through all variables in our input file

                    # all latitudes, altitudes, spectral bands, for all variables, going through months
                    pinatubo_field = src[var][:, :, :, idx]

                    # for background at same month
                    background = bg[var][:, :, :, idx]

                    # we want to only strengthen pinatubo where it is actually higher than 2000-background. tiny detail but worth having in
                    anomaly = pinatubo_field - background

                    # avoid the negative anomalies : we only want to strengthen the pinatubo signal, not the background. 
                    anomaly = xr.where(anomaly > 0.0, anomaly, 0.0)

                    # a long road, but in the end we get: New field = background + weighted (depending on time) x10 pinatubo-anomaly
                    #
                    # when w_total = 1, then we get background + 10x anomaly
                    # background + 10 * anomaly
                    #
                    # when w_total = 0, we only get the background

                    out[var][:, :, :, idx] = background + effective_factor * anomaly

            # write a temporary file first to avoid destroying the original file if writing fails.
            tmp_file = tgt_file.with_suffix(".tmp.nc")
            out.to_netcdf(tmp_file)

        # then overwrite original file if writing was successful. This way we don't risk destroying the original file if writing was unsuccessful. 
        tmp_file.replace(tgt_file)

        print(f"{src_year} -> {tgt_year} done")
        print()